# Customer Churn Prediction using Machine Learning
### An End-to-End Professional Machine Learning Case Study

**Author**: AI/ML Career Candidate  
**Target Roles**: AI/ML Engineer, Data Analyst, Associate Consultant (TCS Digital/Prime)  

---

## 1. Project Overview & Business Understanding

### The Business Problem
Customer churn occurs when a subscriber cancels their service with a company. For a telecom company, acquiring new customers is **5 to 25 times more expensive** than retaining existing ones. By predicting which customers are likely to churn, the marketing team can trigger proactive retention campaigns (e.g., promotional discounts, contract upgrades, specialized support).

### Objectives
1. **Analyze** historical customer data from the Kaggle Telco Customer Churn dataset to uncover drivers of churn.
2. **Implement** a robust data preprocessing and feature engineering pipeline to avoid data leakage.
3. **Train & Compare** multiple classification algorithms: Logistic Regression, Decision Tree, Random Forest, and XGBoost.
4. **Tune** the top model using Stratified Cross-Validation.
5. **Interpret** features using relative importance and formulate actionable business retention recommendations.

### Key Interview Highlight: Metric Selection
In customer churn, **Recall is the critical metric**. Why?
- **False Negative (FN)**: We predict a customer will *not* churn, but they do. This is a massive business loss (loss of monthly recurring revenue, high acquisition cost to replace them).
- **False Positive (FP)**: We predict a customer *will* churn, but they remain loyal. We spend a small amount sending them a loyalty offer. While slightly redundant, this does not hurt and may actually improve retention.

Therefore, we aim to maximize **Recall** and **F1-Score** rather than pure Accuracy.

## 2. Setup & Environment Configurations

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

# Set Seaborn style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Data Path
DATA_PATH = os.path.join('..', 'data', 'raw', 'WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Load Data
if not os.path.exists(DATA_PATH):
    # Fallback to local workspace if run inside the directory
    DATA_PATH = 'data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
    if not os.path.exists(DATA_PATH):
        DATA_PATH = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded with shape: {df.shape}")
df.head()

## 3. Data Cleaning

### TCS Interview Concept: The `TotalCharges` Mystery
**Q**: Why is the column `TotalCharges` loaded as an `object` (string) in Pandas when it should be numeric?
**A**: The dataset contains 11 entries where `tenure` is `0` (new customers). For these rows, the value of `TotalCharges` is recorded as a blank space (`" "`). Because of these spaces, Pandas reads the entire column as a string column. We must replace these blank spaces with `NaN`, convert the column to float, and then logically fill them with `0.0` (as these are brand-new customers).

In [ ]:
# Inspect data types and missing values
print(df.info())

# 1. Clean TotalCharges whitespace
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill new customers (tenure = 0) with 0.0
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

# 2. Convert Target 'Churn' to binary integer
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Confirm cleaning
print("\n--- After Cleaning ---")
print(df[['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']].info())
print(f"Missing values remaining: {df.isna().sum().sum()}")

## 4. Exploratory Data Analysis (EDA)

Let's visualize the major drivers of customer churn.

In [ ]:
# 1. Churn Distribution (Class Imbalance)
plt.figure(figsize=(6, 5))
sns.countplot(x='Churn', data=df, palette={0: '#1e3d59', 1: '#ff6e40'}, hue='Churn', legend=False)
plt.title('Customer Churn Class Distribution')
plt.xticks([0, 1], ['Loyal (No Churn)', 'Churned (Churn)'])
plt.ylabel('Count')
plt.show()

In [ ]:
# 2. Tenure vs Churn
plt.figure(figsize=(10, 5))
sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True, common_norm=False, palette={0: '#1e3d59', 1: '#ff6e40'}, alpha=0.6)
plt.title('Distribution of Customer Tenure by Churn Status')
plt.xlabel('Tenure (Months)')
plt.ylabel('Density')
plt.show()
print("Insight: Customers are highly vulnerable to churn in their first 1-12 months. After 24 months, churn drops drastically.")

In [ ]:
# 3. Contract Type vs Churn
contract_churn = df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack() * 100
ax = contract_churn.plot(kind='bar', stacked=True, color=['#1e3d59', '#ff6e40'], edgecolor='black')
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', label_type='center', color='white', fontweight='bold')
plt.title('Churn Rate by Contract Type')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(['Loyal', 'Churned'], bbox_to_anchor=(1.05, 1))
plt.show()
print("Insight: Month-to-month contracts have an extremely high churn rate (~43%), whereas 2-year contracts are very safe (~3.%).")

In [ ]:
# 4. Internet Service vs Churn
internet_churn = df.groupby('InternetService')['Churn'].value_counts(normalize=True).unstack() * 100
ax = internet_churn.plot(kind='bar', stacked=True, color=['#1e3d59', '#ff6e40'], edgecolor='black')
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', label_type='center', color='white', fontweight='bold')
plt.title('Churn Rate by Internet Service Type')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(['Loyal', 'Churned'], bbox_to_anchor=(1.05, 1))
plt.show()
print("Insight: Fiber Optic customers have high churn rates (~42%), indicating potential service friction or high pricing frustration.")

## 5. Feature Engineering & Train-Test Split

To prevent **Data Leakage**, we split our dataset before fitting any scaling or encoding preprocessors.

In [ ]:
# Feature Engineering Functions
def get_tenure_group(months):
    if months <= 12:
        return '0-1 Year'
    elif months <= 24:
        return '1-2 Years'
    elif months <= 48:
        return '2-4 Years'
    elif months <= 60:
        return '4-5 Years'
    else:
        return 'Over 5 Years'

df['tenure_group'] = df['tenure'].apply(get_tenure_group)

# Count services used
services = ['PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
service_count = np.zeros(len(df))
for service in services:
    service_count += (df[service] == 'Yes').astype(int)
df['Number_of_Services'] = service_count

# Family Status interaction
df['Has_Partner_and_Dependents'] = ((df['Partner'] == 'Yes') & (df['Dependents'] == 'Yes')).astype(int)

# Charge ratio
df['Monthly_to_Total_Ratio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1e-5)

# Separate Features and Target
X = df.drop(columns=['Churn', 'customerID'])
y = df['Churn']

# Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

### Building the Pipeline using Scikit-learn `ColumnTransformer`

In [ ]:
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Number_of_Services', 'Monthly_to_Total_Ratio']
categorical_cols = [col for col in X_train.columns if col not in numerical_cols]

# Construct Transformer pipelines
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, numerical_cols),
    ('cat', cat_pipeline, categorical_cols)
])

# Fit & Transform
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print(f"Processed Train dimensions: {X_train_proc.shape}")
print(f"First feature name: {feature_names[0]}")

## 6. Model Training & Comparison

In [ ]:
# Train baseline models
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', max_depth=8, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_proc, y_train)
    preds = model.predict(X_test_proc)
    probs = model.predict_proba(X_test_proc)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, probs)
    })

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))

## 7. Hyperparameter Tuning with GridSearchCV

We tune a Random Forest classifier using Grid Search with Stratified 5-Fold Cross-Validation.

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [6, 8, 10],
    'min_samples_split': [2, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_proc, y_train)
best_rf = grid.best_estimator_

# Evaluate Tuned Model
preds_tuned = best_rf.predict(X_test_proc)
probs_tuned = best_rf.predict_proba(X_test_proc)[:, 1]

print(f"\nBest Grid parameters: {grid.best_params_}")
print(f"Tuned Random Forest F1: {f1_score(y_test, preds_tuned):.4f}")
print(f"Tuned Random Forest Recall: {recall_score(y_test, preds_tuned):.4f}")

## 8. Feature Importance Interpretation

In [ ]:
importances = best_rf.feature_importances_
feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False).head(15)

sns.barplot(x='Importance', y='Feature', data=feat_imp, palette='viridis', hue='Feature', legend=False)
plt.title('Top 15 Most Important Features - Customer Churn Model')
plt.show()

## 9. TCS / Placements Interview Prep - Quick Q&A

**Q1. What is Data Leakage and how did you prevent it in your project?**
*Answer*: Data Leakage occurs when information from outside the training dataset is used to build the model, causing overly optimistic validation scores but poor real-world testing. In this project, I prevented data leakage by:  
1. Performing train-test split *before* doing any preprocessing, feature scaling (`StandardScaler`), or encoding (`OneHotEncoder`).  
2. Fitting all preprocessors *only* on the training set (`X_train`) and merely transforming the test set (`X_test`).

**Q2. How do you handle highly imbalanced datasets?**
*Answer*: In our dataset, only ~26% of customers churn, creating a class imbalance. I handled this by:  
1. Using `stratify=y` during the train-test split to ensure representative ratios of churn in both sets.  
2. Utilizing class-weight parameters (`class_weight='balanced'`) in Random Forest/Logistic Regression models, or setting `scale_pos_weight` in XGBoost, to penalize errors on minority classes heavily.  
3. Evaluating models on **F1-Score and ROC-AUC** instead of Accuracy, which is biased under skew.

**Q3. How does a Random Forest Classifier work?**
*Answer*: Random Forest is an ensemble bagging method that builds multiple independent Decision Trees on bootstrapped samples of the dataset. For splits, it considers a random subset of features. The final prediction is a majority vote of all trees, which dramatically reduces overfitting and variance compared to a single decision tree.